<a href="https://colab.research.google.com/github/emanameen9982773/Crime-Type-Classification-Based-on-Incident-Features/blob/main/CSC463_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

dataset = pd.read_csv("Crimes_-_2001_to_Present.csv",
                      nrows=500000,
                      on_bad_lines='skip',
                      low_memory=False)

dataset = dataset[dataset['Year'] >= 2015]
dataset = dataset.dropna()



dataset = dataset[['Year', 'District', 'Primary Type', 'Arrest', 'Domestic',
                   'Location Description', 'Beat']]

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
dataset['Primary Type'] = le.fit_transform(dataset['Primary Type'])
class_names = le.classes_

dataset['Arrest'] = dataset['Arrest'].astype(str).str.lower().map({'true': 1, 'false': 0})
dataset['Domestic'] = dataset['Domestic'].astype(str).str.lower().map({'true': 1, 'false': 0})

dataset = pd.get_dummies(dataset, columns=['Location Description'])

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
dataset['Primary Type'] = le.fit_transform(dataset['Primary Type'])

dataset.to_csv("cleaned_dataset.csv", index=False)

dataset.head()

,Year,District,Primary Type,Arrest,Domestic,Beat,Location Description_ABANDONED BUILDING,Location Description_AIRCRAFT,Location Description_AIRPORT BUILDING NON-TERMINAL - NON-SECURE AREA,Location Description_AIRPORT BUILDING NON-TERMINAL - SECURE AREA,...,Location Description_VACANT LOT / LAND,Location Description_VACANT LOT/LAND,Location Description_VEHICLE - COMMERCIAL,Location Description_VEHICLE - DELIVERY TRUCK,Location Description_VEHICLE - OTHER RIDE SERVICE,"Location Description_VEHICLE - OTHER RIDE SHARE SERVICE (E.G., UBER, LYFT)","Location Description_VEHICLE - OTHER RIDE SHARE SERVICE (LYFT, UBER, ETC.)",Location Description_VEHICLE NON-COMMERCIAL,Location Description_VEHICLE-COMMERCIAL,Location Description_WAREHOUSE
0,2015,9,2,0,1,924,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,2015,15,32,0,0,1511,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,2015,14,18,1,0,1412,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,2015,15,1,0,1,1522,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
5,2015,6,3,0,0,614,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [2]:
from sklearn.model_selection import train_test_split

X = dataset.drop('Primary Type', axis=1)
y = dataset['Primary Type']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

Training set: (395084, 134)
Testing set: (98772, 134)


In [3]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import pandas as pd

# Decision Tree
dt_model = DecisionTreeClassifier()
dt_model.fit(X_train, y_train)

y_pred_dt = dt_model.predict(X_test)
dt_accuracy = accuracy_score(y_test, y_pred_dt)

actual_names = le.inverse_transform(y_test.values)
pred_dt_names = le.inverse_transform(y_pred_dt)

dt_results = pd.DataFrame({
    'Actual': actual_names,
    'Predicted_DT': pred_dt_names
})

print("\nDecision Tree Results:")
print(dt_results.head(15))


# Random Forest
rf_model = RandomForestClassifier()
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
rf_accuracy = accuracy_score(y_test, y_pred_rf)

pred_rf_names = le.inverse_transform(y_pred_rf)

rf_results = pd.DataFrame({
    'Actual': actual_names,
    'Predicted_RF': pred_rf_names
})

print("\nRandom Forest Results:")
print(rf_results.head(15))





print("\n=== Accuracy ===")
print("Decision Tree:", dt_accuracy)
print("Random Forest:", rf_accuracy)



Decision Tree Results:
    Actual  Predicted_DT
0       18             1
1       32            32
2        2             2
3       25             6
4        6             2
5       32             1
6        6             2
7       28            18
8       23            23
9       29             2
10       2            32
11       9             6
12       1             2
13      26            18
14      32             6

Random Forest Results:
    Actual  Predicted_RF
0       18             1
1       32            32
2        2             2
3       25             6
4        6            18
5       32            32
6        6             2
7       28            18
8       23            23
9       29             2
10       2            32
11       9             6
12       1             2
13      26            18
14      32             6

=== Accuracy ===
Decision Tree: 0.3867897784797311
Random Forest: 0.3912039849350018


In [4]:

!pip install xgboost

In [5]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
import pandas as pd

#  XGBoost
xgb_model = XGBClassifier(
    use_label_encoder=False,
    eval_metric='mlogloss',
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='hist'
)

xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)
xgb_accuracy = accuracy_score(y_test, y_pred_xgb)

actual_names = le.inverse_transform(y_test.values)
pred_xgb_names = le.inverse_transform(y_pred_xgb)


xgb_results = pd.DataFrame({
    'Actual': actual_names,
    'Predicted_XGB': pred_xgb_names
})

print("\nXGBoost Results:")
print(xgb_results.head(15))

print("\nXGBoost Accuracy:", xgb_accuracy)

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [12:36:10] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



XGBoost Results:
    Actual  Predicted_XGB
0       18             18
1       32             32
2        2              2
3       25              6
4        6             18
5       32             32
6        6              2
7       28             18
8       23             32
9       29              2
10       2              2
11       9              6
12       1              2
13      26             18
14      32             32

XGBoost Accuracy: 0.41651480176568256
